In [2]:
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [3]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset-missouri:latest")
artifact_dir = artifact.download()

df = pl.read_parquet(f"{artifact_dir}/flood_model_missouri.parquet").to_pandas()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\zurek\_netrc.


wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (2549.4MB/s)


In [4]:
df_new = df

In [ ]:
print(df_new.columns)
print(len(df_new.columns))
print(df_new['observation_hour'].head())


Index(['site_id', 'observation_hour', 'latitude', 'longitude',
       'streamflow_cfs_mean', 'streamflow_cfs_max', 'streamflow_cfs_min',
       'gage_height_ft_mean', 'gage_height_ft_max', 'gage_height_ft_min',
       'observation_count', 'precipitation_mm', 'temperature_c',
       'wind_speed_ms', 'specific_humidity_kgkg', 'surface_pressure_pa',
       'shortwave_radiation_wm2', 'longwave_radiation_wm2',
       'potential_evaporation_mm', 'cape_jkg', 'convective_precip_fraction',
       'station_name', 'huc_code', 'drainage_area_sq_km',
       'is_reference_hcdn2009', 'elev_mean_m', 'elev_max_m', 'elev_min_m',
       'SLOPE_PCT', 'ASPECT_NORTHNESS', 'ASPECT_EASTNESS',
       'geology_class_reedbush', 'geology_desc_hunt', 'p_mean', 'pet_mean',
       'aridity_index', 'p_seasonality', 'frac_snow', 'high_prec_freq',
       'low_prec_freq', 'hydroatlas_elev_m', 'hydroatlas_slope_deg',
       'hydroatlas_temp_mean_c', 'hydroatlas_precip_mm_yr',
       'hydroatlas_pet_mm_yr', 'hydroatlas_ar

In [6]:
site_locations = df_new.groupby(["station_name","site_id"]).agg({
    "latitude": 'first',
    "longitude": 'first'
}).reset_index()
print(site_locations.head())

                            station_name   site_id   latitude  longitude
0        Adair Creek at Independence, MO  06893830  39.037778 -94.363333
1         Auxvasse Creek near Reform, MO  06927240  38.760000 -91.840278
2          Big Creek near Blairstown, MO  06921720  38.554978 -93.965353
3     Big Piney River near Big Piney, MO  06930000  37.665639 -92.049917
4  Big Piney below Fort Leonard Wood, MO  06930060  37.760250 -92.057972


In [7]:
fig = px.scatter_mapbox(
    site_locations,
    lat="latitude",
    lon="longitude",
    hover_name="site_id",
    hover_data=["station_name"],
    zoom=4,
    height=800,
    title="All Sites Map"
)

fig.update_layout(
    mapbox_style="open-street-map"
)
fig.show()

C:\Users\zurek\AppData\Local\Temp\ipykernel_12124\580902013.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [8]:
scaler = StandardScaler()
numeric_cols = df_new.select_dtypes(include='number').columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['latitude', 'longitude']]
df_scaled = df_new.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_new[numeric_cols])
std_by_site = (
    df_scaled
    .groupby('site_id')[numeric_cols]
    .std()
    .reset_index()
)

print(std_by_site)

     site_id  streamflow_cfs_mean  streamflow_cfs_max  streamflow_cfs_min  \
0   06820500             1.183702            0.242679            0.237777   
1   06821080             0.132182            0.027902            0.025786   
2   06821150             0.175180            0.036326            0.034805   
3   06893150             0.154509            2.403595            2.393980   
4   06893390             0.179476            0.039720            0.033294   
..       ...                  ...                 ...                 ...   
87  06935955             0.039345            3.275196            3.262133   
88  06935980             0.017469            3.132298            3.119819   
89  06935997             0.007118            0.001846            0.001083   
90  06936475             0.086065            0.019052            0.015949   
91  06936530             0.002316            0.000642            0.000348   

    gage_height_ft_mean  gage_height_ft_max  gage_height_ft_min  \
0       

In [9]:
heatmap_df = std_by_site.set_index('site_id')
fig = px.imshow(
    heatmap_df,
    aspect='auto',
    color_continuous_scale='Viridis',
    labels=dict(
        x="Variable",
        y="Site ID",
        color="Std Dev"
    )
)

fig.update_traces(
    hovertemplate=
        "Site: %{y}<br>" +
        "Variable: %{x}<br>" +
        "Std Dev: %{z:.4f}<extra></extra>"
)

fig.update_layout(
    height=2000,
    yaxis=dict(automargin=True)
)

fig.show()

In [10]:
numeric_cols = df_new.select_dtypes(include=['number']).columns.tolist()
corr_df = df_new[numeric_cols].corr()
corr_df = corr_df.dropna(how='all').dropna(axis=1, how='all')
print(corr_df)

                            latitude  longitude  streamflow_cfs_mean  \
latitude                    1.000000  -0.167136             0.029518   
longitude                  -0.167136   1.000000            -0.005615   
streamflow_cfs_mean         0.029518  -0.005615             1.000000   
streamflow_cfs_max          0.006543  -0.007154             0.999895   
streamflow_cfs_min          0.006298  -0.007037             0.999890   
gage_height_ft_mean         0.131748  -0.375417             0.253892   
gage_height_ft_max         -0.001876  -0.009158             0.255037   
gage_height_ft_min         -0.001847  -0.009115             0.252683   
observation_count          -0.056431   0.717370            -0.078367   
precipitation_mm           -0.008370   0.000785             0.065328   
temperature_c              -0.051963   0.010540             0.016060   
wind_speed_ms               0.029019  -0.042708             0.012262   
specific_humidity_kgkg     -0.056633   0.020526             0.04

In [12]:
corr_pd = corr_df

n_vars = len(corr_pd.columns)

fig_size = max(600, n_vars * 25)

fig = px.imshow(
    corr_pd,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto"
)

fig.update_layout(
    title="Correlation Heatmap",
    width=fig_size,
    height=fig_size,
    xaxis=dict(
        tickangle=45,
        automargin=True
    ),
    yaxis=dict(
        automargin=True
    )
)

fig.update_xaxes(side="bottom")

fig.show()

In [16]:
def pull_wandb(file_name: str,file_path: str = None,n_rows: int | None = None) -> pl.DataFrame:
    run = wandb.init(
        project="flood-forecasting",
        entity="connorjsmith28-rice-university",
        job_type="preprocessing"
    )
    artifact = run.use_artifact(
        f"connorjsmith28-rice-university/flood-forecasting/{file_path}:latest"
    )
    artifact_dir = artifact.download()
    return pl.read_parquet(
        f"{artifact_dir}/{file_name}.parquet",
        n_rows=n_rows,)
config = {"n_rows": 100,
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri"
}

In [17]:
df = pull_wandb(config["file_name"],config["file_path"],config['n_rows'])

wandb: Currently logged in as: cz88 (connorjsmith28-rice-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:26.3 (23.0MB/s)


In [19]:
print(df)

shape: (100, 51)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ site_id  ┆ observati ┆ latitude  ┆ longitude ┆ … ┆ hydroatla ┆ hydroatla ┆ hydroatla ┆ hydroatla │
│ ---      ┆ on_hour   ┆ ---       ┆ ---       ┆   ┆ s_sand_pc ┆ s_forest_ ┆ s_crop_pc ┆ s_urban_p │
│ str      ┆ ---       ┆ f64       ┆ f64       ┆   ┆ t         ┆ pct       ┆ t         ┆ ct        │
│          ┆ datetime[ ┆           ┆           ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│          ┆ μs, UTC]  ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 06923250 ┆ 2008-08-0 ┆ 37.684306 ┆ -92.92463 ┆ … ┆ 34.502088 ┆ 49.937232 ┆ 36.317987 ┆ 0.90795   │
│          ┆ 3         ┆           ┆ 9         ┆   ┆           ┆           ┆           ┆           │
│          ┆ 21:00:00  ┆           ┆           ┆   ┆           ┆          